# 18.11 教師的風格和錯誤會被學走嗎？

# 第 18 章：蒸餾，讓小模型向教師學習

前置：CE、模型輸出與學生訓練。教師像經驗較多的老師：可以只給答案，也可展示對各候選的分配。老師可能說錯；學生也須真的選較小架構。此處只驗loss與梯度，正式蒸餾需讀者提供已訓練教師。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：老師不只說第一名，也指出接近的候選**

學生選較小架構；老師可能教錯，仍要真值與行爲評估。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/distillation.svg")))

**先想一想：**老師每題都拒絕，學生模仿成功是任務成功嗎？

老師的風格、錯誤與過度拒絕都可能被學生模仿。先檢查生成資料，再分別評價內容、格式、安全與正常完成，不只比相似度。

**把直覺寫成數學：**教師一致率、標準答案正確率、行爲指標獨立列。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
teacher_answers = ["不知道", "不知道", "不知道"]
truth = ["2", "4", "資訊不足"]
student_answers = teacher_answers.copy()
print("教師一致率", sum(a == b for a, b in zip(teacher_answers, student_answers)) / 3)
print("示例真值exact match", sum(a == b for a, b in zip(truth, student_answers)) / 3)

**如何讀結果：**構造例只說明一致與正確不同。真值中的不確定可接受措辭需人工rubric，不能只exact match。

**只改一件事：**只修正教師一個已知錯誤回答。
